<h1>Bibliotecas</h1>
<i> - Coleta de dados validados como boatos = falso

In [5]:
import os
import time
import requests
import pandas as pd

from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

<h1>Google Fact Check</h1>

<h2>Consulta - API</h2>

In [6]:
# ===============================
# CARREGAR CHAVE DA API
# ===============================

load_dotenv("../google-factcheck-api-key.env")

API_KEY = os.getenv("GOOGLE_FACTCHECK_API_KEY")

if not API_KEY:
    raise ValueError("Chave da API não encontrada. Verifique o arquivo google-factcheck-api-key.env")

# ===============================
# CONFIGURAÇÕES DA API
# ===============================

URL = "https://factchecktools.googleapis.com/v1alpha1/claims:search"

termos_busca = [
    "urnas eletrônicas",
    "fraude nas urnas",
    "TSE",
    "eleições 2022",
    "eleições 2026",
    "voto impresso",
    "urna auditável",

    "Lula",
    "Bolsonaro",
    "Alexandre de Moraes",
    "Moraes",
    "Renan Santos",
    "MBL",

    "INSS",
    "fraude INSS",
    "Pix imposto",
    "taxação do Pix",
    "anistia Bolsonaro",
    "PL da anistia",
    "STF redes sociais",
    "Congresso Nacional"
]

page_size = 50
max_age_days = 730
max_paginas_por_termo = 3

# ===============================
# COLETA
# ===============================

registros = []

for termo in termos_busca:

    print(f"\nConsultando termo: {termo}")

    page_token = None
    pagina_atual = 1

    while pagina_atual <= max_paginas_por_termo:

        params = {
            "query": termo,
            "languageCode": "pt",
            "pageSize": page_size,
            "maxAgeDays": max_age_days,
            "key": API_KEY
        }

        if page_token:
            params["pageToken"] = page_token

        resposta = requests.get(URL, params=params)

        if resposta.status_code != 200:
            print(f"Erro ao buscar '{termo}': {resposta.status_code}")
            print(resposta.text[:500])
            break

        dados = resposta.json()

        claims = dados.get("claims", [])

        print(f"Página {pagina_atual} - claims encontradas: {len(claims)}")

        for claim in claims:

            texto_claim = claim.get("text", "")
            data_claim = claim.get("claimDate", "")

            for review in claim.get("claimReview", []):

                registros.append({
                    "termo_busca": termo,
                    "texto_afirmacao": texto_claim,
                    "data_claim": data_claim,
                    "fonte_verificacao": review.get("publisher", {}).get("name", ""),
                    "url_checagem": review.get("url", ""),
                    "avaliacao_original": review.get("textualRating", ""),
                    "data_publicacao": review.get("reviewDate", ""),
                    "url_consulta": resposta.url,
                    "data_coleta": datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
                    "origem_pipeline": "GOOGLE_FACTCHECK"
                })

        page_token = dados.get("nextPageToken")

        if not page_token:
            break

        pagina_atual += 1

        time.sleep(0.5)

# ===============================
# DATAFRAME RAW
# ===============================

df_google_raw = pd.DataFrame(registros)

print(f"\nTotal de registros coletados: {len(df_google_raw)}")

df_google_raw.head()


Consultando termo: urnas eletrônicas
Página 1 - claims encontradas: 43

Consultando termo: fraude nas urnas
Página 1 - claims encontradas: 42

Consultando termo: TSE
Página 1 - claims encontradas: 47

Consultando termo: eleições 2022
Página 1 - claims encontradas: 15

Consultando termo: eleições 2026
Página 1 - claims encontradas: 20

Consultando termo: voto impresso
Página 1 - claims encontradas: 10

Consultando termo: urna auditável
Página 1 - claims encontradas: 2

Consultando termo: Lula
Página 1 - claims encontradas: 50
Página 2 - claims encontradas: 50
Página 3 - claims encontradas: 50

Consultando termo: Bolsonaro
Página 1 - claims encontradas: 50
Página 2 - claims encontradas: 50
Página 3 - claims encontradas: 50

Consultando termo: Alexandre de Moraes
Página 1 - claims encontradas: 50
Página 2 - claims encontradas: 50
Página 3 - claims encontradas: 50

Consultando termo: Moraes
Página 1 - claims encontradas: 50
Página 2 - claims encontradas: 50
Página 3 - claims encontradas: 

,termo_busca,texto_afirmacao,data_claim,fonte_verificacao,url_checagem,avaliacao_original,data_publicacao,url_consulta,data_coleta,origem_pipeline
0,urnas eletrônicas,"Pilili, mascote da urna eletrônica, faz o L de...",,BOL - UOL,https://www.bol.uol.com.br/noticias/2026/05/08...,Falso,,https://factchecktools.googleapis.com/v1alpha1...,10/05/2026 02:42:00,GOOGLE_FACTCHECK
1,urnas eletrônicas,"Pilili, mascote da urna eletrônica, faz o L de...",,UOL Notícias,https://noticias.uol.com.br/confere/ultimas-no...,Falso,,https://factchecktools.googleapis.com/v1alpha1...,10/05/2026 02:42:00,GOOGLE_FACTCHECK
2,urnas eletrônicas,"Foto mostra a mascote ""Pilili"", do TSE, fazend...",2026-05-05T00:00:00Z,AFP Checamos,https://checamos.afp.com/doc.afp.com.B2A88ZQ,Falso,2026-05-08T17:21:00Z,https://factchecktools.googleapis.com/v1alpha1...,10/05/2026 02:42:00,GOOGLE_FACTCHECK
3,urnas eletrônicas,Lula perdeu todas as eleições com votação manu...,2025-12-07T00:00:00Z,AFP Checamos,https://checamos.afp.com/doc.afp.com.88868T3,Enganoso,2025-12-15T17:22:00Z,https://factchecktools.googleapis.com/v1alpha1...,10/05/2026 02:42:00,GOOGLE_FACTCHECK
4,urnas eletrônicas,Lula perdeu todas as eleições feitas com cédul...,2025-12-09T00:00:00Z,Aos Fatos,https://www.aosfatos.org/noticias/falso-que-lu...,falso,2025-12-09T00:00:00Z,https://factchecktools.googleapis.com/v1alpha1...,10/05/2026 02:42:00,GOOGLE_FACTCHECK


<h2>Extração</h2>

In [7]:
nome_pipeline = "pipeline_falso_google_factcheck"
nome_base = "google_factcheck"

df_exportar = df_google_raw

data_agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

pasta_raw = Path(f"../dados/{nome_pipeline}/raw")
pasta_raw.mkdir(parents=True, exist_ok=True)

caminho_saida = pasta_raw / f"{nome_base}_raw_{data_agora}.csv"

df_exportar.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

print(f"Arquivo bruto salvo em: {caminho_saida}")
print(f"\nTotal de registros extraídos: {len(df_exportar)}")
print(f"\nData e hora da extração: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

Arquivo bruto salvo em: ..\dados\pipeline_falso_google_factcheck\raw\google_factcheck_raw_2026-05-10_02-42-39.csv

Total de registros extraídos: 1021

Data e hora da extração: 10/05/2026 02:42:39
